In [ ]:
# @title 1. 环境初始化 (Clean Install)
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. 设定根目录
ROOT = Path("/content")
os.chdir(ROOT)

# 2. 清理并重新克隆仓库 (确保代码纯净)
repo_dir = ROOT / "motion-diffusion-model"
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("📂 克隆仓库中...")
subprocess.run(["git", "clone", "https://github.com/GuyTevet/motion-diffusion-model.git"], check=True)
os.chdir(repo_dir)

# 3. 安装依赖 (强制指定兼容版本)
print("📦 安装依赖库...")
# 卸载冲突包
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "scikit-image", "imageio", "matplotlib"], check=False)

# 安装修复版依赖
# matplotlib==3.7.3: 修复 MoviePy 报错
# imageio==2.33.0: 修复 scikit-image 报错
pkgs = [
    "imageio==2.33.0", 
    "scikit-image==0.22.0", 
    "matplotlib==3.7.3",
    "git+https://github.com/openai/CLIP.git",
    "smplx", "chumpy", "trimesh", "moviepy", "gradio", "gdown"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)

# 3.1 修复 chumpy 与 Python 3.12 的兼容性 (getargspec)
import sysconfig
site_packages = next((p for p in sys.path if 'site-packages' in p), None)
if site_packages:
    for rel in ['chumpy/ch.py', 'chumpy/linalg.py']:
        target = Path(site_packages) / rel
        if target.exists():
            text = target.read_text()
            if 'getfullargspec' not in text:
                target.write_text(text.replace('inspect.getargspec', 'inspect.getfullargspec'))
                print(f'🔧 已修复 {rel} (getargspec -> getfullargspec)')
            else:
                print(f'ℹ️ {rel} 已兼容，跳过。')
        else:
            print(f'⚠️ 未找到 {rel}，请确认 chumpy 安装。')
else:
    print('⚠️ 未找到 site-packages 目录，chumpy 可能未安装。')


# 3.2 在所有 Python 进程中强制兼容 (sitecustomize)
if site_packages:
    sitecustomize = Path(site_packages) / 'sitecustomize.py'
    sitecustomize.write_text("""import inspect\nif not hasattr(inspect, 'getargspec'):\n    inspect.getargspec = inspect.getfullargspec  # type: ignore[attr-defined]\n""")
    print(f'🔧 已写入 {sitecustomize} 以兼容 getargspec 缺失。')
else:
    print('⚠️ 未写入 sitecustomize.py，因为未找到 site-packages。')

# 4. 系统级依赖
subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=False)
os.environ["SDL_AUDIODRIVER"] = "dummy"

print("✅ 环境准备就绪。")

In [ ]:
# @title 2. 下载核心资产 (SMPL & Stats)
import os
import zipfile
import subprocess
from pathlib import Path
from google.colab import files

# --- A. 下载 SMPL 模型 (解决视频空白) ---
smpl_dir = Path("body_models")
smpl_dir.mkdir(parents=True, exist_ok=True)
target_pkl = smpl_dir / "smpl/SMPL_NEUTRAL.pkl"
zip_name = "smpl.zip"

if not target_pkl.exists():
    print("⬇️ 正在下载 SMPL 模型...")
    # 尝试 gdown
    subprocess.run(["gdown", "--id", "1INYlGA76ak_cKGzvpOV2Pe6RkYTlXTW2", "-O", zip_name], check=False)
    
    if Path(zip_name).exists() and Path(zip_name).stat().st_size > 10 * 1024 * 1024:
        print("📂 解压 SMPL...")
        with zipfile.ZipFile(zip_name, 'r') as zf:
            zf.extractall(smpl_dir)
        os.remove(zip_name)
        print("✅ SMPL 模型自动下载成功。")
    else:
        print("\n" + "!"*60)
        print("❌ Google Drive 自动下载失败 (Quota Exceeded)")
        print("!"*60)
        print("👉 请手动下载 smpl.zip，然后点击下方按钮上传。")
        print("下载地址: https://drive.google.com/uc?id=1INYlGA76ak_cKGzvpOV2Pe6RkYTlXTW2&export=download")
        
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(smpl_dir)
                print("✅ SMPL 模型手动安装成功。")
else:
    print("✅ SMPL 模型已存在。")

# --- B. 下载数据集统计文件 ---
data_root = Path("dataset/HumanML3D")
data_root.mkdir(parents=True, exist_ok=True)
(data_root / "texts").mkdir(exist_ok=True)

print("⬇️ 下载数据集统计 (Mean/Std)...")
base_url = "https://github.com/GuyTevet/motion-diffusion-model/raw/main/dataset"
os.system(f"curl -L -o {data_root}/Mean.npy {base_url}/t2m_mean.npy")
os.system(f"curl -L -o {data_root}/Std.npy {base_url}/t2m_std.npy")

# --- C. 创建伪造数据集 (绕过检测) ---
# 注意：这里使用了严格的 4 段式格式，防止 index error
for split in ["train", "val", "test"]:
    with open(data_root / f"{split}.txt", "w") as f:
        f.write("sample01\nsample02")

dummy_content = "a person is walking#a person is walking#0.0#0.0\n"
with open(data_root / "texts/sample01.txt", "w") as f: f.write(dummy_content)
with open(data_root / "texts/sample02.txt", "w") as f: f.write(dummy_content)

print("✅ 数据集配置完成。")

In [ ]:
# @title 3. 下载预训练模型
import subprocess
import zipfile
from pathlib import Path

save_dir = Path("save")
zip_path = Path("humanml_trans_enc_512.zip")
file_id = "1PE0PK8e5a5j-7-Xhs5YET5U5pGh0c821" 

if not (save_dir / "humanml_trans_enc_512/model000200000.pt").exists():
    print("⬇️ 下载模型权重...")
    subprocess.run(["gdown", "--id", file_id, "-O", str(zip_path)], check=False)
    
    if zip_path.exists() and zip_path.stat().st_size > 100 * 1024 * 1024:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(save_dir)
        zip_path.unlink()
        print("✅ 模型下载成功。")
    else:
        print("❌ 模型下载失败，请手动下载 humanml_trans_enc_512.zip 并上传到 save/ 目录")
else:
    print("✅ 模型已存在。")

In [ ]:
# @title 4. 彻底修复 IndentationError
from pathlib import Path

target_file = Path("data_loaders/humanml/data/dataset.py")
print(f"🔧 正在修复 {target_file} ...")

if target_file.exists():
    with open(target_file, "r") as f:
        lines = f.readlines()
    
    new_lines = []
    skip_counter = 0 # 计数器，用于连续注释多行
    
    for line in lines:
        # 如果计数器大于0，说明处于要注释的块中
        if skip_counter > 0:
            new_lines.append(f"        # {line.lstrip()}") 
            skip_counter -= 1
            continue
            
        # 检测到导致报错的断言起始行
        if "assert len(self.t2m_dataset) > 1" in line:
            print("   ✅ 发现断言块，正在注释整块代码(4行)...")
            new_lines.append(f"        # {line.lstrip()}")
            skip_counter = 3 # 接下来3行也必须注释掉，否则会报 IndentationError
        else:
            new_lines.append(line)
            
    with open(target_file, "w") as f:
        f.writelines(new_lines)
    print("✅ 修复完成：IndentationError 已彻底解决。")
else:
    print("❌ 错误：找不到文件。请检查第 1 步是否成功。")

In [ ]:
# @title 5. 启动网页服务 (Run)
import gradio as gr
import glob
import subprocess
import sys
import os
from pathlib import Path

def generate_motion(text_prompt, motion_length, seed, repetition):
    model_path = "save/humanml_trans_enc_512/model000200000.pt"
    if not os.path.exists(model_path): return None, "❌ 模型缺失，请检查第 3 步。"

    # 清理
    subprocess.run("rm -rf save/humanml_trans_enc_512/samples_*", shell=True)
    
    print(f"🎬 生成: {text_prompt}")
    
    cmd = [
        sys.executable, "-m", "sample.generate",
        "--model_path", model_path,
        "--text_prompt", text_prompt,
        "--motion_length", str(motion_length),
        "--seed", str(int(seed)),
        "--num_repetitions", str(int(repetition)),
        "--device", "0"
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        # 查找视频
        mp4_files = list(Path("save/humanml_trans_enc_512").glob("samples_*/**/*.mp4"))
        
        if not mp4_files:
            return None, f"❌ 未生成视频。\n日志:\n{result.stdout}\n错误:\n{result.stderr}"
            
        return str(max(mp4_files, key=os.path.getctime)), "✅ 成功！"
    except Exception as e:
        return None, str(e)

iface = gr.Interface(
    fn=generate_motion,
    inputs=[
        gr.Textbox(label="Prompt", value="a person is dancing"),
        gr.Slider(1, 10, value=3, label="Duration"),
        gr.Number(value=42, label="Seed"),
        gr.Slider(1, 3, value=1, step=1, label="Repetitions")
    ],
    outputs=[gr.Video(), gr.Textbox()],
    title="MDM Web Service (Final Fixed)",
    description="如果生成视频为空白，请回到第 2 步手动上传 SMPL 模型。"
)

iface.launch(share=True, debug=False)